In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 4: 모델 학습, Experiment Tracking 및 Model Registry

Module 3에서 생성한 **불변 Dataset** (`customer_ltv_training@v1`)으로 모델을 학습하고,
Model Registry에 등록합니다.

### 학습 목표
- Dataset 객체에서 데이터를 읽어 모델 학습
- XGBoost vs Random Forest 성능 비교
- Experiment Tracking으로 실험 관리
- Model Registry에 Champion/Challenger 등록


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.ensemble import RandomForestRegressor
from snowflake.ml.registry import Registry
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml import dataset

from sklearn.metrics import mean_squared_error as _mse, r2_score as _r2

session = get_active_session()
print('Session ready')


## 1. Dataset에서 학습 데이터 로드

Module 3에서 `generate_dataset()`으로 생성한 불변 Dataset을 로드합니다.
이 Dataset은 Point-in-Time JOIN이 적용되어 Data Leakage가 없습니다.


In [ ]:
# 불변 Dataset 로드
training_ds = dataset.load_dataset(session, 'customer_ltv_training', 'v1')
df = training_ds.read.to_snowpark_dataframe()

print(f'Training data: {df.count():,} rows')
print(f'Columns: {df.columns}')
df.show(5)


In [ ]:
# 피처/타겟 정의
LABEL_COL = 'FUTURE_12M_REVENUE'
CATEGORICAL_COLS = ['C_MKTSEGMENT']
NUMERIC_COLS = ['C_ACCTBAL', 'TOTAL_ORDERS', 'AVG_ORDER_VALUE',
                'TOTAL_REVENUE', 'AVG_DISCOUNT', 'AVG_QUANTITY',
                'DAYS_SINCE_LAST_ORDER', 'C_NATIONKEY']
ALL_FEATURES = CATEGORICAL_COLS + NUMERIC_COLS

print(f'Features ({len(ALL_FEATURES)}): {ALL_FEATURES}')
print(f'Target: {LABEL_COL}')


## 2. Train/Test 분할


In [ ]:
train_df, test_df = df.random_split([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')


## 3. Experiment Tracking 설정


In [ ]:
# Experiment 생성 (DEMO.ML_DEMO 스키마에 저장)
session.sql("USE DATABASE DEMO").collect()
session.sql("USE SCHEMA ML_DEMO").collect()

exp = ExperimentTracking(session=session)
exp.set_experiment("customer_ltv_regression")
print("Experiment ready: customer_ltv_regression")


## 4. 모델 학습: XGBoost Baseline


In [ ]:
# XGBoost Baseline
# 반복 실행 대응: 기존 run 삭제
try:
    exp.delete_run('xgboost_baseline')
except Exception:
    pass
with exp.start_run('xgboost_baseline'):
    pipeline_xgb = Pipeline(steps=[
        ('encoder', OrdinalEncoder(
            input_cols=CATEGORICAL_COLS, output_cols=CATEGORICAL_COLS,
            handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler(
            input_cols=NUMERIC_COLS, output_cols=NUMERIC_COLS)),
        ('regressor', XGBRegressor(
            input_cols=ALL_FEATURES, label_cols=[LABEL_COL],
            output_cols=['PREDICTED_LABEL'],
            n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42))
    ])
    pipeline_xgb.fit(train_df)

    # Evaluate
    preds = pipeline_xgb.predict(test_df).select(LABEL_COL, 'PREDICTED_LABEL').to_pandas()
    rmse1 = float(np.sqrt(_mse(preds[LABEL_COL], preds['PREDICTED_LABEL'])))
    r2_1 = float(_r2(preds[LABEL_COL], preds['PREDICTED_LABEL']))

    exp.log_param('model', 'XGBRegressor')
    exp.log_param('n_estimators', 100)
    exp.log_param('max_depth', 6)
    exp.log_metric('rmse', rmse1)
    exp.log_metric('r2', r2_1)

print(f'XGBoost Baseline — RMSE: {rmse1:,.0f}, R²: {r2_1:.4f}')


## 5. 모델 학습: XGBoost Tuned


In [ ]:
# XGBoost Tuned
# 반복 실행 대응: 기존 run 삭제
try:
    exp.delete_run('xgboost_tuned')
except Exception:
    pass
with exp.start_run('xgboost_tuned'):
    pipeline_xgb2 = Pipeline(steps=[
        ('encoder', OrdinalEncoder(
            input_cols=CATEGORICAL_COLS, output_cols=CATEGORICAL_COLS,
            handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler(
            input_cols=NUMERIC_COLS, output_cols=NUMERIC_COLS)),
        ('regressor', XGBRegressor(
            input_cols=ALL_FEATURES, label_cols=[LABEL_COL],
            output_cols=['PREDICTED_LABEL'],
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=42))
    ])
    pipeline_xgb2.fit(train_df)

    preds = pipeline_xgb2.predict(test_df).select(LABEL_COL, 'PREDICTED_LABEL').to_pandas()
    rmse2 = float(np.sqrt(_mse(preds[LABEL_COL], preds['PREDICTED_LABEL'])))
    r2_2 = float(_r2(preds[LABEL_COL], preds['PREDICTED_LABEL']))

    exp.log_param('model', 'XGBRegressor')
    exp.log_param('n_estimators', 200)
    exp.log_param('max_depth', 4)
    exp.log_metric('rmse', rmse2)
    exp.log_metric('r2', r2_2)

print(f'XGBoost Tuned — RMSE: {rmse2:,.0f}, R²: {r2_2:.4f}')


## 6. 모델 학습: Random Forest


In [ ]:
# Random Forest
# 반복 실행 대응: 기존 run 삭제
try:
    exp.delete_run('random_forest')
except Exception:
    pass
with exp.start_run('random_forest'):
    pipeline_rf = Pipeline(steps=[
        ('encoder', OrdinalEncoder(
            input_cols=CATEGORICAL_COLS, output_cols=CATEGORICAL_COLS,
            handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler(
            input_cols=NUMERIC_COLS, output_cols=NUMERIC_COLS)),
        ('regressor', RandomForestRegressor(
            input_cols=ALL_FEATURES, label_cols=[LABEL_COL],
            output_cols=['PREDICTED_LABEL'],
            n_estimators=150, max_depth=10, random_state=42))
    ])
    pipeline_rf.fit(train_df)

    preds = pipeline_rf.predict(test_df).select(LABEL_COL, 'PREDICTED_LABEL').to_pandas()
    rmse3 = float(np.sqrt(_mse(preds[LABEL_COL], preds['PREDICTED_LABEL'])))
    r2_3 = float(_r2(preds[LABEL_COL], preds['PREDICTED_LABEL']))

    exp.log_param('model', 'RandomForestRegressor')
    exp.log_param('n_estimators', 150)
    exp.log_param('max_depth', 10)
    exp.log_metric('rmse', rmse3)
    exp.log_metric('r2', r2_3)

print(f'Random Forest — RMSE: {rmse3:,.0f}, R²: {r2_3:.4f}')


## 7. 실험 결과 비교


In [ ]:
# 결과 비교
results = pd.DataFrame({
    'Model': ['XGBoost Baseline', 'XGBoost Tuned', 'Random Forest'],
    'RMSE': [rmse1, rmse2, rmse3],
    'R2': [r2_1, r2_2, r2_3]
})
print(results.to_string(index=False))

# Best model selection (lowest RMSE)
best_idx = results['RMSE'].idxmin()
best_run_name = results.loc[best_idx, 'Model']
best_pipeline = [pipeline_xgb, pipeline_xgb2, pipeline_rf][best_idx]
print(f'\nBest model: {best_run_name} (RMSE: {results.loc[best_idx, "RMSE"]:,.0f})')


## 8. Feature Importance


In [ ]:
# Feature Importance 추출
best_step = dict(best_pipeline.steps)['regressor']
try:
    model = best_step.to_xgboost()
except Exception:
    model = best_step.to_sklearn()

fi_df = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
top = fi_df.head(9).sort_values('Importance')
ax.barh(top['Feature'], top['Importance'], color='#636EFA')
ax.set_title(f'Feature Importance — {best_run_name}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()


## 9. Model Registry 등록

Best 모델을 V1 (Champion), 차순위 모델을 V2 (Challenger)로 등록합니다.


In [ ]:
# Registry 초기화
reg = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")

# 반복 실행 대응: Inference Service 및 기존 모델 삭제
session.sql("DROP SERVICE IF EXISTS DEMO.ML_DEMO.LTV_INFERENCE_SVC").collect()
session.sql("DROP MODEL IF EXISTS DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR").collect()
print("기존 서비스/모델 정리 완료")

# V1: Best model (Champion)
mv_v1 = reg.log_model(
    model=best_pipeline,
    model_name="CUSTOMER_LTV_PREDICTOR",
    version_name="V1",
    sample_input_data=train_df.select(ALL_FEATURES + [LABEL_COL]),
    comment=f"V1 Champion: {best_run_name}"
)
mv_v1.set_alias("champion")
print(f"V1 registered: {best_run_name} (Champion)")


In [ ]:
# V2: Challenger (Champion 다음으로 성능 좋은 모델)
challenger_pipeline = pipeline_xgb2 if best_idx != 1 else pipeline_rf
challenger_name = 'XGBoost Tuned' if best_idx != 1 else 'Random Forest'

mv_v2 = reg.log_model(
    model=challenger_pipeline,
    model_name='CUSTOMER_LTV_PREDICTOR',
    version_name='V2',
    sample_input_data=train_df.select(ALL_FEATURES + [LABEL_COL]),
    comment=f'V2 Challenger: {challenger_name}'
)
mv_v2.set_alias('challenger')
print(f'V2 registered: {challenger_name} (Challenger)')


In [ ]:
# Registry 확인
print('=== Model Versions ===')
model = reg.get_model('CUSTOMER_LTV_PREDICTOR')
for v in model.versions():
    print(f'  {v.version_name}: {v.comment}')

print(f'\nChampion: {model.default.version_name}')


## 정리

| 단계 | 내용 |
|------|------|
| 데이터 소스 | `customer_ltv_training@v1` (불변 Dataset, Point-in-Time) |
| 실험 비교 | XGBoost Baseline / Tuned / Random Forest |
| Best 모델 | RMSE 최소 기준 자동 선택 |
| Registry | V1=Champion, V2=Challenger |

### 다음 단계
Module 5에서 등록된 모델로 추론합니다.
